In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [4]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 68.1 MB/s eta 0:00:00:00:0100:01


In [5]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

print("Creating knowledge base...")

kb = []

for _, row in train.iterrows():
    correct_letter = row["answer"]
    kb.append(str(row[correct_letter]))

print(f"Knowledge base size: {len(kb)}")

Creating knowledge base...
Knowledge base size: 2000


In [7]:
print("Loading embedding model...")

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Creating embeddings...")

kb_embeddings = model.encode(
    kb,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embedding shape:", kb_embeddings.shape)

index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created!")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Creating embeddings...


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Embedding shape: (2000, 384)
Knowledge base successfully created!


In [8]:
zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

row_150 = train.iloc[150]

prompt_150 = str(row_150["prompt"])

labels_150 = [
    str(row_150["A"]),
    str(row_150["B"]),
    str(row_150["C"]),
    str(row_150["D"]),
    str(row_150["E"])
]

ans_150 = str(row_150[row_150["answer"]])

print(prompt_150)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.


In [9]:
result = zs(
    prompt_150,
    candidate_labels=labels_150,
    multi_label=False
)

print(result)

{'sequence': 'Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.', 'labels': ['The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states that would have followed without the alteration, as defined by Einstein in his book "The concept of Relativity."', 'The butterfly effect is the phenomenon that a small change in the initial conditio

In [10]:
correct_text = row_150[row_150["answer"]]

correct_score = result["scores"][result["labels"].index(correct_text)]

print("Correct option:", row_150["answer"])
print("Correct answer text:", correct_text)
print("Score:", round(correct_score, 3))

Correct option: C
Correct answer text: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
Score: 0.384


In [11]:
query_embedding = model.encode(
    [prompt_150],
    convert_to_numpy=True
)

k = 10

distances, indices = index.search(query_embedding, k)

print(indices)

[[ 663 1701 1269 1532  576  847 1693 1906  168  150]]


In [12]:
retrieved = indices[0]

rank = np.where(retrieved == 150)[0]

if len(rank) > 0:
    print("Rank:", rank[0] + 1)
else:
    print("Not found in Top-10")

Rank: 10


In [13]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

retrieved_indices = indices[0]

docs_10 = [kb[i] for i in retrieved_indices]

pairs = [[prompt_150, doc] for doc in docs_10]

ce_scores = cross_encoder.predict(pairs)

ranking = sorted(
    zip(retrieved_indices, ce_scores),
    key=lambda x: x[1],
    reverse=True
)

for rank, (doc_idx, score) in enumerate(ranking, start=1):
    print(f"Rank {rank}: KB Index = {doc_idx}, Score = {score:.4f}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Rank 1: KB Index = 150, Score = 4.7585
Rank 2: KB Index = 847, Score = 4.7526
Rank 3: KB Index = 1693, Score = 4.7526
Rank 4: KB Index = 1906, Score = 4.7526
Rank 5: KB Index = 1269, Score = 4.7375
Rank 6: KB Index = 1532, Score = 4.7375
Rank 7: KB Index = 168, Score = 4.7072
Rank 8: KB Index = 576, Score = 4.6870
Rank 9: KB Index = 663, Score = 4.6602
Rank 10: KB Index = 1701, Score = 4.6602


In [14]:
for rank, (doc_idx, score) in enumerate(ranking, start=1):
    if doc_idx == 150:
        print("True document rank:", rank)
        break

True document rank: 1


In [15]:
row_42 = train.iloc[42]

prompt_42 = str(row_42["prompt"])

query_embedding = model.encode(
    [prompt_42],
    convert_to_numpy=True
)

k = 5

distances, indices = index.search(query_embedding, k)

retrieved_indices = indices[0]

docs_5 = [kb[i] for i in retrieved_indices]

concatenated_docs = " ".join(docs_5)

text = f"Context: {concatenated_docs} Question: {prompt_42}"

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(len(tokenizer.encode(text, truncation=False)))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

216


In [16]:
true_document = kb[150]

rag_prompt = f"Context: {true_document} Question: {prompt_150}"

result_rag = zs(
    rag_prompt,
    candidate_labels=labels_150,
    multi_label=False
)

correct_text = row_150[row_150["answer"]]

new_score = result_rag["scores"][
    result_rag["labels"].index(correct_text)
]

print("Correct answer:", correct_text)
print("New score:", round(new_score, 3))

Correct answer: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
New score: 0.989


In [17]:
wrong_document = kb[999]

adversarial_prompt = f"Context: {wrong_document} Question: {prompt_150}"

result_adv = zs(
    adversarial_prompt,
    candidate_labels=labels_150,
    multi_label=False
)

correct_text = row_150[row_150["answer"]]

adv_score = result_adv["scores"][
    result_adv["labels"].index(correct_text)
]

print("Correct answer:", correct_text)
print("Probability:", round(adv_score, 3))

Correct answer: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
Probability: 0.529


In [18]:
hits = 0
k = 5

for i in range(100):
    row = train.iloc[i]

    prompt = str(row["prompt"])

    # Correct answer text
    correct_answer = str(row[row["answer"]])

    # Embed the prompt
    query_embedding = model.encode(
        [prompt],
        convert_to_numpy=True
    )

    # Retrieve top-5 documents
    distances, indices = index.search(query_embedding, k)

    retrieved_docs = [kb[idx] for idx in indices[0]]

    # Check whether the correct answer text appears in any retrieved document
    if correct_answer in retrieved_docs:
        hits += 1

hit_rate = (hits / 100) * 100

print("Hits:", hits)
print("Hit Rate:", round(hit_rate, 1))

Hits: 73
Hit Rate: 73.0


In [19]:
def map_at_3(actual, predicted):
    if actual in predicted:
        return 1 / (predicted.index(actual) + 1)
    return 0.0


scores = []

k = 5

for i in range(20):

    row = train.iloc[i]

    prompt = str(row["prompt"])
    correct_letter = row["answer"]

    # Option texts
    labels = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]

    # Retrieve
    query_embedding = model.encode(
        [prompt],
        convert_to_numpy=True
    )

    distances, indices = index.search(query_embedding, k)

    retrieved_indices = indices[0]
    docs = [kb[idx] for idx in retrieved_indices]

    # Rerank
    pairs = [[prompt, doc] for doc in docs]

    ce_scores = cross_encoder.predict(pairs)

    best_doc = docs[np.argmax(ce_scores)]

    # Augment
    rag_prompt = f"Context: {best_doc} Question: {prompt}"

    # Predict
    result = zs(
        rag_prompt,
        candidate_labels=labels,
        multi_label=False
    )

    # Convert predicted texts back to option letters
    text_to_letter = {
        str(row["A"]): "A",
        str(row["B"]): "B",
        str(row["C"]): "C",
        str(row["D"]): "D",
        str(row["E"]): "E",
    }

    predicted_letters = [
        text_to_letter[label]
        for label in result["labels"][:3]
    ]

    score = map_at_3(correct_letter, predicted_letters)

    scores.append(score)

final_map3 = np.mean(scores)

print("MAP@3:", round(final_map3, 3))

MAP@3: 0.975
